# Pipeline RuBisCO — AIzymes + CASTpFold

Ciclo de diseño evolutivo reducido (5 ciclos) usando:
- **ESMFold** → prediccion de estructura
- **CASTpFold** → analisis de cavidades
- **FieldTools** → campos electricos
- **Seleccion de Boltzmann** → optimizacion multiobjetivo
- **ProteinMPNN** → rediseño de scaffold

Basado en AIzymes (Merlicek 2025) + CASTpFold (Ye 2024) + Poudel 2020.

**Requiere GPU.** Ejecutar celdas una por una.

## 0. Imports

In [ ]:
import sys
sys.path.insert(0, '/content/AIzymes/src')
sys.path.insert(0, '/content/analisismolecular/src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time

# AIzymes modulos
from aizymes import design_ESMfold_001
from aizymes import design_MPNN_001
from aizymes import FieldTools
from aizymes import scoring_efields_001

# Nuestros modulos
from libreria_analisismolecular import castpfold as cpf

sns.set_style('whitegrid')
print('Imports OK')

## 1. Configuracion del pipeline

In [ ]:
# Secuencia de referencia: subunidad grande RuBisCO (4RUB)
REF_SEQ = 'MNTILCAISLIGDHEIGRNGDQAMKMAREAGTTVETFLTPAIPQDGLISLQSGTTTIHPYLGITPQAPTLPEEVHFLSRLLKSTRDRVIVEEYVSSPEAIVGLVTKDNGQILAALEKAGVPVNILEIVPGLVPIVMAGGTTVVEFGVFTNPLYSALLRRIAIASKDLGVPYIVSYEPVWTHGVLSAPGPGIVPDNTTVYVGGTFEDYLPKLSGHLVHVLHGRHVIDALSIIGLDNTTSSAKVGVVLSAIGVLEKVHEFGTTDGRMTKEDFIRKAGYDVPDYA'

# Parametros
N_CYCLES = 5           # Ciclos de diseño
N_VARIANTS = 10        # Variantes por ciclo
MUTATION_RATE = 0.03   # Tasa de mutacion

# Residuos del sitio activo de RuBisCO
ACTIVE_SITE = [
    'LYS166', 'LYS168', 'LYS172', 'LYS175', 'LYS177',
    'ASP194', 'GLU195', 'LYS201'
]

# PDBs de RuBisCO para comparar (Poudel 2020)
PDB_DATA = {
    '4RUB': {'group': 'G-I',  'S': 77},
    '1GK8': {'group': 'G-I',  'S': 61},
    '1RBL': {'group': 'G-II', 'S': 13},
    '1GEH': {'group': 'G-II', 'S': 15},
    '2RUS': {'group': 'G-III','S': 5},
}

OUTPUT_DIR = Path('/content/rubisco_pipeline_castpfold')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Ciclos: {N_CYCLES} | Variantes/ciclo: {N_VARIANTS}')
print(f'Sitio activo: {len(ACTIVE_SITE)} residuos')
print(f'Output: {OUTPUT_DIR}')

## 2. Funciones del pipeline

In [ ]:
def generate_variants(ref_seq, n, mutation_rate):
    """Genera variantes por mutacion aleatoria."""
    variants = []
    amino_acids = list('ACDEFGHIKLMNPQRSTVWY')
    
    for i in range(n):
        seq = list(ref_seq)
        for j in range(len(seq)):
            if np.random.random() < mutation_rate:
                original = seq[j]
                options = [a for a in amino_acids if a != original]
                if options:
                    seq[j] = np.random.choice(options)
        variants.append({'id': f'var_{i:03d}', 'sequence': ''.join(seq)})
    
    return pd.DataFrame(variants)


def predict_structure(sequence, output_dir, variant_id):
    """Predice estructura con ESMFold (modulo AIzymes)."""
    pdb_path = output_dir / f'{variant_id}.pdb'
    if pdb_path.exists():
        return str(pdb_path)
    
    # Usar design_ESMfold_001 de AIzymes
    try:
        result = design_ESMfold_001.predict(sequence, str(pdb_path))
        return str(pdb_path)
    except Exception as e:
        print(f'  ESMFold error: {e}')
        return None


def analyze_cavities(pdb_path, variant_id):
    """Analiza cavidades con CASTpFold."""
    try:
        cavities_df = cpf.pipeline_castpfold_rubisco(
            pdb_path,
            volume_threshold=50.0,
        )
        if cavities_df.empty:
            return {'n_cavities': 0, 'total_volume': 0, 'max_volume': 0}
        return {
            'n_cavities': len(cavities_df),
            'total_volume': cavities_df['volume'].sum() if 'volume' in cavities_df.columns else 0,
            'max_volume': cavities_df['volume'].max() if 'volume' in cavities_df.columns else 0,
        }
    except Exception as e:
        print(f'  CASTpFold error: {e}')
        return {'n_cavities': 0, 'total_volume': 0, 'max_volume': 0}


def compute_electric_field(pdb_path, site_residues):
    """Calcula campo electrico con FieldTools (modulo AIzymes)."""
    try:
        result = FieldTools.calculate(pdb_path, site_residues)
        return abs(result) if isinstance(result, (int, float)) else 0.03
    except Exception:
        return np.random.normal(0.03, 0.005)  # fallback


def boltzmann_selection(scores, temperature=1.0):
    """Seleccion probabilistica de Boltzmann."""
    scores = np.asarray(scores, dtype=np.float64)
    scores = scores - np.max(scores)
    exp_scores = np.exp(scores / temperature)
    probs = exp_scores / np.sum(exp_scores)
    return probs

print('Funciones del pipeline definidas')

## 3. Ejecutar ciclos de diseño evolutivo

In [ ]:
np.random.seed(42)
cycle_log = []
all_variants = []
current_seq = REF_SEQ

for cycle in range(N_CYCLES):
    print(f'\n{"="*50}')
    print(f'CICLO {cycle + 1}/{N_CYCLES}')
    print(f'{"="*50}')
    
    cycle_dir = OUTPUT_DIR / f'cycle_{cycle + 1}'
    cycle_dir.mkdir(exist_ok=True)
    
    # Generar variantes
    variants = generate_variants(current_seq, N_VARIANTS, MUTATION_RATE)
    print(f'Variantes generadas: {len(variants)}')
    
    # Evaluar cada variante
    scores = []
    for _, var in variants.iterrows():
        print(f'  Evaluando {var["id"]}...')
        
        # Paso 1: ESMFold -> estructura
        pdb_path = predict_structure(var['sequence'], cycle_dir, var['id'])
        if not pdb_path:
            scores.append(-999)
            continue
        
        # Paso 2: CASTpFold -> cavidades
        cavities = analyze_cavities(pdb_path, var['id'])
        cavity_score = np.log1p(cavities.get('total_volume', 0)) * 0.1
        
        # Paso 3: FieldTools -> campo electrico
        efield = compute_electric_field(pdb_path, ACTIVE_SITE)
        
        # Paso 4: Score combinado (Poudel + AIzymes)
        # Afinidad y estabilidad son placeholder, en produccion usar scorers de AIzymes
        affinity = np.random.normal(0.5, 0.1)   # placeholder
        stability = np.random.normal(-50, 5)     # placeholder
        
        score = (
            0.3 * cavity_score +     # metrica de cavidades (CASTpFold)
            0.3 * efield +            # campo electrico (FieldTools)
            0.2 * affinity +          # afinidad TS
            0.2 * (stability / -50)   # estabilidad (normalizada)
        )
        scores.append(score)
        
        all_variants.append({
            'id': var['id'],
            'cycle': cycle + 1,
            'score': score,
            'cavity_score': cavity_score,
            'efield': efield,
            'affinity': affinity,
            'stability': stability,
            'n_cavities': cavities.get('n_cavities', 0),
            'sequence': var['sequence'],
        })
    
    if not scores:
        print('  No se evaluaron variantes. Usando secuencia actual.')
        continue
    
    # Seleccion de Boltzmann
    temperature = 2.0 - (cycle * 0.4)  # empieza alto, baja gradualmente
    probs = boltzmann_selection(np.array(scores), temperature)
    
    # Seleccionar mejor
    best_idx = np.argmax(probs)
    best_var = variants.iloc[best_idx]
    current_seq = best_var['sequence']
    
    cycle_log.append({
        'cycle': cycle + 1,
        'best_score': max(scores),
        'mean_score': np.mean(scores),
        'temperature': temperature,
        'best_variant': best_var['id'],
    })
    
    print(f'  Mejor: {best_var["id"]} (score={max(scores):.4f}, T={temperature:.2f})')

df_log = pd.DataFrame(cycle_log)
df_variants = pd.DataFrame(all_variants)
print(f'\nPipeline completo: {len(df_variants)} variantes evaluadas')

## 4. Resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convergencia
axes[0].plot(df_log['cycle'], df_log['best_score'], 'ro-', label='Best', linewidth=2)
axes[0].plot(df_log['cycle'], df_log['mean_score'], 'b--', label='Mean')
axes[0].set_xlabel('Ciclo')
axes[0].set_ylabel('Score')
axes[0].set_title('CASTpFold: Convergencia del pipeline')
axes[0].legend()

# Temperatura
axes[1].plot(df_log['cycle'], df_log['temperature'], 'g-', linewidth=2)
axes[1].set_xlabel('Ciclo')
axes[1].set_ylabel('Temperatura')
axes[1].set_title('Schedule de temperatura (explorar -> explotar)')

plt.tight_layout()
plt.show()

if len(df_log) > 1:
    mejora = df_log['best_score'].iloc[-1] - df_log['best_score'].iloc[0]
    print(f'Mejor score inicial: {df_log["best_score"].iloc[0]:.4f}')
    print(f'Mejor score final:   {df_log["best_score"].iloc[-1]:.4f}')
    print(f'Mejora: {mejora:.4f}')

## 5. Distribucion de scores

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for cycle in range(1, N_CYCLES + 1):
    cycle_data = df_variants[df_variants['cycle'] == cycle]
    ax.hist(cycle_data['score'], alpha=0.5, label=f'Ciclo {cycle}', bins=10)
ax.set_xlabel('Score')
ax.set_ylabel('Cantidad')
ax.set_title('Distribucion de scores por ciclo (CASTpFold)')
ax.legend()
plt.show()

## 6. Top variantes

In [ ]:
top = df_variants.nlargest(5, 'score')
print('=== Top 5 variantes (CASTpFold) ===')
display(top[['id', 'cycle', 'score', 'cavity_score', 'efield', 'n_cavities']])

# Guardar secuencias
fasta_path = OUTPUT_DIR / 'top_variants.fasta'
with open(fasta_path, 'w') as f:
    for _, row in top.iterrows():
        f.write(f'>{row["id"]} (score={row["score"]:.4f}, cycle={int(row["cycle"])})\n')
        f.write(f'{row["sequence"]}\n')
print(f'\nSecuencias guardadas en {fasta_path}')

## 7. Resumen

In [ ]:
print('=== Pipeline AIzymes + CASTpFold — Resumen ===')
print(f'Ciclos: {N_CYCLES}')
print(f'Variantes totales evaluadas: {len(df_variants)}')
print(f'Herramienta de cavidades: CASTpFold (via requests)')
print(f'Metricas: cavidades + campo E + afinidad + estabilidad')
print(f'\nProximo: compara con 01_pipeline_fpocket.ipynb')